In [ ]:
import nest_asyncio
nest_asyncio.apply()

import numpy as np
import plotly.graph_objects as go
from ipywidgets import FloatSlider, HBox, VBox, ToggleButton
import asyncio
from IPython.display import display, HTML

class HORSimulation:
    def __init__(self):
        self.reset()
        
    def reset(self):
        self.t_data = [0.0]
        self.p_data = [0.0]
        self.theta_oh_data = [0.0]
        self.theta_free_data = [1.0]

        self.t = 0.0
        self.potential = 0.0
        self.theta_oh = 0.0
        self.theta_co = 0.0
        self.theta_free = 1.0

        self.dt = 0.003

    def step(self, current_val, co_val):
        j_target = current_val / 100.
        co_press = co_val / 150.0
        C_dl = 4
        theta_free = max(1e-6, 1 - self.theta_co - self.theta_oh)

        d_theta_co = 50.0 * co_press * theta_free
        rate_oh_formation = 8.0 * np.exp((self.potential - 0.6) / 0.033) * theta_free
        r_clean = 20.0 * self.theta_co * self.theta_oh

        self.theta_co = np.clip(self.theta_co + (d_theta_co - r_clean) * self.dt, 0, 1)
        self.theta_oh = np.clip(self.theta_oh + (rate_oh_formation - r_clean) * self.dt, 0, 1)
        self.theta_free = max(1e-6, 1 - self.theta_co - self.theta_oh)

        j_hor = 0.2 * self.theta_free * np.sinh(self.potential / 0.08)
        dE_dt = (j_target - j_hor) / C_dl
        self.potential = np.clip(self.potential + dE_dt * self.dt, 0, 1)

        self.t += self.dt
        self.t_data.append(self.t)
        self.p_data.append(self.potential)
        self.theta_oh_data.append(self.theta_oh)
        self.theta_free_data.append(self.theta_free)

        if len(self.t_data) > 6650:
            self.t_data.pop(0)
            self.p_data.pop(0)
            self.theta_oh_data.pop(0)
            self.theta_free_data.pop(0)

sim = HORSimulation()

fig = go.FigureWidget()

fig.add_scatter(
    x=sim.t_data,
    y=sim.p_data,
    mode='lines',
    line=dict(color='black', width=2),
    name='Potential',
    yaxis='y1'
)

fig.add_scatter(
    x=sim.t_data,
    y=[value * 100 for value in sim.theta_oh_data],
    mode='lines',
    line=dict(color='red', width=2),
    name='Θ (OH)',
    yaxis='y2'
)

fig.add_scatter(
    x=sim.t_data,
    y=[value * 100 for value in sim.theta_free_data],
    mode='lines',
    line=dict(color='blue', width=2),
    name='Θ (Free)',
    yaxis='y2'
)

fig.update_layout(
    xaxis=dict(title=dict(text='Time', font=dict(size=18, color='black')), showticklabels=False, linecolor='black', linewidth=2, mirror=True, showline=True, showgrid=False, range=[0, 20]),
    yaxis=dict(
        title=dict(text='Potential (V)', font=dict(size=18, color='black')),
        range=[0, 1.0],
        showgrid=False,
        linecolor='black',
        linewidth=2,
        mirror=True,
        showline=True,
        ticks='outside',
        tickwidth=2,
        tickfont=dict(size=16, color='black')
    ),
    yaxis2=dict(
        title=dict(text='Surface coverage (%)', font=dict(size=18, color='black')),
        autorange=True,
        fixedrange=False,
        overlaying='y',
        side='right',
        showgrid=False,
        ticks='outside',
        tickwidth=2,
        tickfont=dict(size=16, color='black')
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(x=0.01, y=0.99, font=dict(size=16, color='black')),
    margin=dict(l=50, r=50, t=50, b=50)
)

slider_layout = dict(width='350px', height='50px')
slider_style = {'description_width': 'initial'}

slider_co = FloatSlider(
    value=60, min=0, max=100, step=1,
    description='CO-Concentration:',
    layout=slider_layout,
    style=slider_style,
    readout_format='d'
)

slider_curr = FloatSlider(
    value=50, min=0, max=100, step=1,
    description='Current:',
    layout=slider_layout,
    style=slider_style,
    readout_format='d'
)

play_btn = ToggleButton(
    value=False,
    description='▶ Start',
    button_style='success',
    layout=dict(width='700px', height='60px')
)

display(HTML('''
    <style>
        .widget-label {
            font-size: 16px !important;
        }
        .widget-readout {
            font-size: 18px !important;
            color: #000000 !important;
        }
        .jupyter-button {
            font-size: 20px !important;
            font-weight: bold !important;
        }
        .slider-container {
            padding-top: 10px;
        }
    </style>
'''))

async def update_loop():
    while play_btn.value:
        # 1. Simulation berechnen (bleibt physikalisch exakt)
        for _ in range(60):
            sim.step(slider_curr.value, slider_co.value)

        # 2. Daten für die Anzeige reduzieren (Downsampling)
        # [::10] bedeutet: nimm nur jeden 10. Punkt aus der Liste
        show_t = sim.t_data[::10]
        show_p = sim.p_data[::10]
        show_oh = [v * 100 for v in sim.theta_oh_data[::10]]
        show_free = [v * 100 for v in sim.theta_free_data[::10]]

        last_t = sim.t_data[-1]

        # 3. Grafik-Update (jetzt viel leichter für das Handy)
        with fig.batch_update():
            fig.data[0].x = show_t
            fig.data[0].y = show_p
            fig.data[1].x = show_t
            fig.data[1].y = show_oh
            fig.data[2].x = show_t
            fig.data[2].y = show_free
            
            fig.layout.xaxis.range = [max(0, last_t - 20), max(20, last_t)]
            # Fixiere yaxis2 autorange, falls es zu sehr springt:
            # fig.layout.yaxis2.range = [0, 105] 

        # 0.05s oder 0.06s ist ein guter Kompromiss für Mobile
        await asyncio.sleep(0.055)

def on_play_change(change):
    if change['new']:
        play_btn.description = '⏸ Stop'
        play_btn.button_style = 'danger'
        asyncio.create_task(update_loop())
    else:
        play_btn.description = '▶ Start'
        play_btn.button_style = 'success'

play_btn.observe(on_play_change, names='value')

display(VBox([HBox([slider_co, slider_curr]), play_btn, fig]))